# Limpiar imágenes y videos sin uso de la tesis LaTeX

Este cuaderno revisa el proyecto, detecta las imágenes que **no** aparecen en ningún `.tex` y las mueve a una carpeta aparte. También mueve los archivos **fuente** que generaron esas imágenes (por ejemplo un `.pptx`), reconociéndolos por tener el mismo nombre.

**Cómo funciona en pocas palabras.** Primero lee todos los `.tex` y anota qué nombres de imagen se mencionan. Luego busca las imágenes que existen en las carpetas del proyecto. Una imagen se considera *sin uso* solo si su nombre no aparece en ningún `.tex`. Esto no depende de qué comando la incluya, así que funciona igual con `\includegraphics` y con tus macros propias como `\utemFigurePropia` o `\addutem`.

**Seguridad.** Por defecto está en modo simulación: primero te muestra qué movería y no toca nada. Cuando estés conforme, cambias una opción y vuelves a ejecutar. Además guarda un registro para poder deshacer todo con la última celda.

**Cómo usarlo.** Deja este cuaderno dentro de la carpeta del proyecto (donde está `main.tex`) o indica la ruta en la celda de configuración. Luego ejecuta las celdas de arriba hacia abajo.

## 1. Configuración

Es la única celda que necesitas ajustar.

In [42]:
from pathlib import Path

# Carpeta raíz del proyecto (donde está main.tex).
# Déjalo en None para que la busque sola desde la ubicación de este cuaderno.
PROJECT_ROOT = None

# Extensiones que LaTeX incluye como imagen.
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".gif", ".bmp", ".tif", ".tiff", ".eps", ".svg", ".pdf", ".webp", ".mp4"}
# Nota. El .pdf NO está en la lista a propósito, para no tocar el PDF compilado de la tesis.
#       Si usas figuras guardadas en PDF, agrégalo arriba con cuidado.

# Extensiones de los archivos fuente que generan las imágenes (editables).
SOURCE_EXTS = {".pptx", ".ppt", ".key", ".drawio", ".vsdx", ".ai",
               ".psd", ".xcf", ".fig", ".odp", ".pub"}

# Carpeta destino donde se juntará todo lo que no se usa.
UNUSED_DIRNAME = "_sin_uso"

# Carpetas que NO se revisan (salidas de compilación, control de versiones, etc.).
EXCLUDE_DIRS = {"build", ".git", ".github", "__pycache__", UNUSED_DIRNAME}

# En True solo muestra el plan y no mueve nada. Cambia a False para mover de verdad.
DRY_RUN = False

# En True ignora las referencias comentadas con % en el LaTeX.
# Una imagen que solo aparece comentada no sale en el PDF, así que cuenta como no usada.
IGNORE_COMMENTS = True

# En True mueve también los archivos fuente que no tienen ninguna imagen asociada
# en el proyecto (huérfanos). Por defecto en False para no tocar lo que no pediste.
MOVE_ORPHAN_SOURCES = False

## 2. Localizar la raíz y preparar ayudantes

In [43]:
import re

def encontrar_raiz(inicio: Path) -> Path:
    inicio = inicio.resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "main.tex").exists():
            return carpeta
    return inicio  # si no aparece main.tex, usa la carpeta actual

ROOT = Path(PROJECT_ROOT).resolve() if PROJECT_ROOT else encontrar_raiz(Path.cwd())
UNUSED_DIR = ROOT / UNUSED_DIRNAME

def esta_excluido(ruta: Path) -> bool:
    return any(parte in EXCLUDE_DIRS for parte in ruta.parts)

def quitar_comentarios(texto: str) -> str:
    # Borra desde un % no escapado hasta el fin de la línea.
    return "\n".join(re.sub(r"(?<!\\)%.*$", "", linea) for linea in texto.splitlines())

print("Raíz del proyecto:", ROOT)
print("Carpeta destino  :", UNUSED_DIR)

Raíz del proyecto: G:\My Drive\doc_tesis
Carpeta destino  : G:\My Drive\doc_tesis\_sin_uso


## 3. Leer los `.tex` y anotar qué imágenes se mencionan

In [44]:
archivos_tex = [p for p in ROOT.rglob("*.tex")
                if not esta_excluido(p.relative_to(ROOT))]

partes = []
for tex in archivos_tex:
    contenido = tex.read_text(encoding="utf-8", errors="ignore")
    if IGNORE_COMMENTS:
        contenido = quitar_comentarios(contenido)
    partes.append(contenido)
texto_total = "\n".join(partes)

# Cualquier texto tipo ruta que termine en una extensión de imagen.
exts_regex = "|".join(sorted(e.lstrip(".") for e in IMAGE_EXTS))
patron_ruta = re.compile(r"[\w\-./\\]+\.(?:%s)" % exts_regex, re.IGNORECASE)

nombres_referenciados = set()   # ej. "fig_lazo_pid.png"
stems_referenciados = set()     # ej. "fig_lazo_pid"  (nombre sin extensión)

for m in patron_ruta.findall(texto_total):
    nombre = Path(m.replace("\\", "/")).name.lower()
    nombres_referenciados.add(nombre)
    stems_referenciados.add(Path(nombre).stem)

# includegraphics escrito sin extensión: {ruta/figura}
for m in re.findall(r"\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}", texto_total):
    nombre = Path(m.replace("\\", "/")).name.lower()
    if Path(nombre).suffix.lower() not in IMAGE_EXTS:
        stems_referenciados.add(nombre)

print("Archivos .tex revisados:", len(archivos_tex))
print("Nombres de imagen referenciados:", len(nombres_referenciados))

Archivos .tex revisados: 16
Nombres de imagen referenciados: 27


## 4. Clasificar las imágenes del proyecto

Una imagen es *usada* si su nombre completo o su nombre sin extensión aparece en algún `.tex`. Todo lo demás queda *sin uso*.

In [45]:
imagenes = [p for p in ROOT.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTS
            and not esta_excluido(p.relative_to(ROOT))]

usadas, sin_uso = [], []
for img in imagenes:
    if img.name.lower() in nombres_referenciados or img.stem.lower() in stems_referenciados:
        usadas.append(img)
    else:
        sin_uso.append(img)

stems_sin_uso = {img.stem.lower() for img in sin_uso}

print("Imágenes en disco:", len(imagenes))
print("  usadas :", len(usadas))
print("  sin uso:", len(sin_uso))

Imágenes en disco: 28
  usadas : 27
  sin uso: 1


## 5. Archivos fuente asociados

Para cada archivo fuente (por ejemplo un `.pptx`) se mira su nombre.

Si su nombre corresponde a una imagen usada, se conserva. Si corresponde a una imagen sin uso, se marca para mover. Si no tiene ninguna imagen asociada, queda como huérfano y solo se mueve si activaste esa opción.

In [46]:
fuentes = [p for p in ROOT.rglob("*")
           if p.is_file()
           and p.suffix.lower() in SOURCE_EXTS
           and not esta_excluido(p.relative_to(ROOT))]

fuentes_a_mover, fuentes_huerfanas, fuentes_en_uso = [], [], []
for f in fuentes:
    stem = f.stem.lower()
    if stem in stems_referenciados:
        fuentes_en_uso.append(f)
    elif stem in stems_sin_uso:
        fuentes_a_mover.append(f)
    else:
        fuentes_huerfanas.append(f)

if MOVE_ORPHAN_SOURCES:
    fuentes_a_mover += fuentes_huerfanas

print("Archivos fuente encontrados:", len(fuentes))
print("  a mover  :", len(fuentes_a_mover))
print("  huérfanos:", len(fuentes_huerfanas), " (se mueven =", MOVE_ORPHAN_SOURCES, ")")
print("  en uso   :", len(fuentes_en_uso))

Archivos fuente encontrados: 3
  a mover  : 0
  huérfanos: 0  (se mueven = False )
  en uso   : 3


## 6. Ver el plan antes de mover

Esta celda solo muestra qué pasaría. No mueve nada.

In [47]:
def rel(p): return p.relative_to(ROOT).as_posix()

print("=" * 60)
print("IMÁGENES SIN USO (se moverán):")
for img in sorted(sin_uso, key=rel):
    print("  -", rel(img))

print()
print("ARCHIVOS FUENTE A MOVER (asociados a imágenes sin uso):")
for f in sorted(fuentes_a_mover, key=rel):
    print("  -", rel(f))

if fuentes_huerfanas:
    print()
    print("FUENTES HUÉRFANAS (sin imagen asociada; no se mueven salvo que actives la opción):")
    for f in sorted(fuentes_huerfanas, key=rel):
        print("  ?", rel(f))

print()
print("=" * 60)
print("Total a mover:", len(sin_uso) + len(fuentes_a_mover), "archivo(s)")
print("Modo:", "SIMULACIÓN, no se mueve nada" if DRY_RUN else "EJECUCIÓN REAL")

IMÁGENES SIN USO (se moverán):
  - 2.Texto/Imag/summitxl.webp

ARCHIVOS FUENTE A MOVER (asociados a imágenes sin uso):

Total a mover: 1 archivo(s)
Modo: EJECUCIÓN REAL


## 7. Mover los archivos

Con `DRY_RUN = True` solo simula. Cuando el plan te parezca correcto, cambia a `DRY_RUN = False` en la celda de configuración, vuelve a ejecutar desde ahí y corre esta celda.

Los archivos se mueven conservando su ruta interna dentro de la carpeta destino, así no se pierden ni chocan entre sí. Además se guarda un registro para poder deshacer.

In [48]:
import csv, shutil
from datetime import datetime

a_mover = sin_uso + fuentes_a_mover

if not a_mover:
    print("No hay archivos que mover.")
elif DRY_RUN:
    print("[SIMULACIÓN] Se moverían", len(a_mover), "archivo(s) a:", UNUSED_DIR)
    print("Cambia DRY_RUN = False en la celda de configuración y vuelve a ejecutar para mover de verdad.")
else:
    UNUSED_DIR.mkdir(parents=True, exist_ok=True)
    manifiesto = UNUSED_DIR / "movimientos.csv"
    filas = []
    for origen in a_mover:
        destino = UNUSED_DIR / origen.relative_to(ROOT)
        destino.parent.mkdir(parents=True, exist_ok=True)
        if destino.exists():
            destino = destino.with_name(destino.stem + "_dup" + destino.suffix)
        shutil.move(str(origen), str(destino))
        filas.append([origen.relative_to(ROOT).as_posix(),
                      destino.relative_to(ROOT).as_posix()])
        print("movido:", origen.relative_to(ROOT).as_posix())

    nuevo = not manifiesto.exists()
    with open(manifiesto, "a", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        if nuevo:
            w.writerow(["fecha", "origen", "destino"])
        ahora = datetime.now().isoformat(timespec="seconds")
        for o, d in filas:
            w.writerow([ahora, o, d])

    print()
    print("Listo.", len(filas), "archivo(s) movido(s).")
    print("Registro guardado en:", manifiesto.relative_to(ROOT).as_posix())

movido: 2.Texto/Imag/summitxl.webp

Listo. 1 archivo(s) movido(s).
Registro guardado en: _sin_uso/movimientos.csv


## 8. Deshacer (opcional)

Devuelve a su lugar todo lo que se movió, leyendo el registro.

In [49]:
# import csv, shutil

# manifiesto = UNUSED_DIR / "movimientos.csv"
# if not manifiesto.exists():
#     print("No hay registro de movimientos para deshacer.")
# else:
#     with open(manifiesto, encoding="utf-8") as fh:
#         filas = list(csv.DictReader(fh))
#     devueltos = 0
#     for fila in filas:
#         destino = ROOT / fila["destino"]
#         origen = ROOT / fila["origen"]
#         if destino.exists():
#             origen.parent.mkdir(parents=True, exist_ok=True)
#             shutil.move(str(destino), str(origen))
#             devueltos += 1
#     print("Devueltos", devueltos, "archivo(s) a su ubicación original.")
#     print("Si ya no lo necesitas, puedes borrar el registro:",
#           manifiesto.relative_to(ROOT).as_posix())